## Enable Local TikToken


### Step 1: Getting the blob URL

First, Grab the tokenizer blob URL from the source on your remote machine. If we trace the get_encoding function, we find it calls a function from tiktoken_ext.openai_public which has the blob URIs for each encoder. Identify the correct function, then print the source

In [1]:
import tiktoken_ext.openai_public
import inspect
import hashlib
import os
import tiktoken


print(dir(tiktoken_ext.openai_public))
# The encoder we want is cl100k_base, we see this as a possible function

print(inspect.getsource(tiktoken_ext.openai_public.cl100k_base))
# The URL should be in the 'load_tiktoken_bpe function call'

print(inspect.getsource(tiktoken_ext.openai_public.gpt2))
# The URL should be in the 'load_tiktoken_bpe function call'

['ENCODING_CONSTRUCTORS', 'ENDOFPROMPT', 'ENDOFTEXT', 'FIM_MIDDLE', 'FIM_PREFIX', 'FIM_SUFFIX', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'cl100k_base', 'data_gym_to_mergeable_bpe_ranks', 'gpt2', 'load_tiktoken_bpe', 'o200k_base', 'p50k_base', 'p50k_edit', 'r50k_base', 'r50k_pat_str']
def cl100k_base():
    mergeable_ranks = load_tiktoken_bpe(
        "https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken",
        expected_hash="223921b76ee99bde995b7ff738513eef100fb51d18c93597a113bcffe865b2a7",
    )
    special_tokens = {
        ENDOFTEXT: 100257,
        FIM_PREFIX: 100258,
        FIM_MIDDLE: 100259,
        FIM_SUFFIX: 100260,
        ENDOFPROMPT: 100276,
    }
    return {
        "name": "cl100k_base",
        "pat_str": r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s""",
        "mergeable_ranks": mergeable_ranks,
       

### Step 3: Copy and rename file

Now, transfer the file to your remote machine to a new folder. Tracing the get_encoding function further reveals a call to tiktoken.load.read_file_cached() which indicates the file needs to be renamed. To get the name for the file, run the following code (pulled from source):

#### Step 3.1 c1100k_base.tiktoken

In [2]:
blobpath = "https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken"
cache_key = hashlib.sha1(blobpath.encode()).hexdigest()
print(cache_key)

9b5ad71b2ce5302211f9c61530b329a4922fc6a4


#### Step 3.2 gpt2

In [3]:
blobpath = "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/vocab.bpe"
cache_key = hashlib.sha1(blobpath.encode()).hexdigest()
print(cache_key)

blobpath = "https://openaipublic.blob.core.windows.net/gpt-2/encodings/main/encoder.json"
cache_key = hashlib.sha1(blobpath.encode()).hexdigest()
print(cache_key)

6d1cbeee0f20b3d9449abfede4726ed8212e3aee
6c7ea1a7e38e3a7f062df639a5b80947f075ffe6


### Step 4: Set up the tiktoken cache

The read_file_cached function then checks environment variables for a cache path and reads from there, so lets set that up:

**Note:** this is not the full path to the tiktoken file, only the path to the folder containing the file

This code snippet will need to be run every time you need tiktoken.

In [14]:
current_dir

'c:/Users/jomedin/Documents/Azure_AI_Services/custom_tiktoken/tiktokencache'

In [13]:
current_dir =os.getcwd().replace("\\", "/") + "/tiktokencache"

tiktoken_cache_dir = current_dir
os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir

# validate
assert os.path.exists(os.path.join(tiktoken_cache_dir, cache_key))
assert os.path.exists(tiktoken_cache_dir)

encoding = tiktoken.get_encoding("cl100k_base")
encoding.encode("Hello, world")

[9906, 11, 1917]

### Step 5: Modyfing TokenEstimator class

In [5]:
from typing import List, Optional, Union

In [6]:
class TokenEstimator(object):

    current_dir =os.getcwd() + "\\tiktokencache"
    tiktoken_cache_dir = current_dir
    os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir

    GPT2_TOKENIZER = tiktoken.get_encoding("gpt2")
    CHATGPT_TOKENIZER = tiktoken.get_encoding("cl100k_base")

    def estimate_tokens(self, text: Union[str, List]) -> int:
        if isinstance(text, str):
            return len(self.GPT2_TOKENIZER.encode(text, allowed_special="all"))
        else:
            # https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
            tokens_per_message = 4  # every message follows <|start|>{role/name}\n{content}<|end|>\n
            tokens_per_name = -1    # if there's a name, the role is omitted
            num_tokens = 0
            for message in text:
                num_tokens += tokens_per_message
                for key, value in message.items():
                    num_tokens += len(self.CHATGPT_TOKENIZER.encode(value))
                    if key == "name":
                        num_tokens += tokens_per_name
            num_tokens += 3  # every reply is primed with <|start|>assistant<|message|>
            return num_tokens

    def construct_tokens_with_size(self, tokens: str, numofTokens: int) -> str:
        newTokens = self.GPT2_TOKENIZER.decode(
            self.GPT2_TOKENIZER.encode(tokens, allowed_special="all")[:numofTokens]
        )
        return newTokens